In [1]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Math, display
sp.init_printing()

In [2]:
def fancy_display(*args, sep=' ', latexer=sp.latex, mather=Math):
    """
    Принимает произвольное количество строк и объектов SymPy.
    Строки добавляет как есть, остальные объекты переводит в LaTeX.
    """
    if mather is None:
        mather = lambda i: i
    if latexer is None:
        latexer = lambda i: str(i)

    parts = []
    for item in args:
        if isinstance(item, str):
            parts.append(item)
        else:
            parts.append(latexer(item))

    full_latex = sep.join(parts)
    display(mather(full_latex))

## Задание 1

> Найти асимптотические линии прямого геликоида.

In [3]:
u, v, t = sp.symbols('u,v,t', real=True)
a, b, c = sp.symbols('a:c', real=True, positive=True)

In [4]:
r = sp.Matrix([
    u * sp.cos(v),
    u * sp.sin(v),
    a * v
])

fancy_display(r'r = ', r, r'.')

<IPython.core.display.Math object>

In [5]:
def get_normal(r):
    """
    Вычисляет нормаль к поверхности в точке (u, v)
    """
    r_u = r.diff(u)
    r_v = r.diff(v)
    
    n = r_u.cross(r_v)
    n /= n.norm()
    return n

def get_I_and_II(r):
    """
    Вычисляет первую и вторую квадратичную формы
    Возвращает коэффициенты этих форм
    """
    r_u = r.diff(u)
    r_v = r.diff(v)
    
    n = get_normal(r)
    
    r_uu = r_u.diff(u)
    r_uv = r_u.diff(v)
    r_vv = r_v.diff(v)

    E = r_u.dot(r_u).simplify()
    F = r_u.dot(r_v).simplify()
    G = r_v.dot(r_v).simplify()
    
    L = r_uu.dot(n).simplify()
    M = r_uv.dot(n).simplify()
    N = r_vv.dot(n).simplify()

    I  = (E, F, G)
    II = (L, M, N)

    return I, II

In [6]:
_, (L, M, N) = get_I_and_II(r)

In [7]:
fancy_display(
    r'\mathbb{I} = ', L, 
    r'\ \mathrm{d}u^2 +', 2 * M, 
    r'\, \mathrm{d}u \mathrm{d}v + ', N, 
    r'\, \mathrm{d}v^2'
)

<IPython.core.display.Math object>

In [8]:
fancy_display(M, r'\dot{u} \dot{v} = 0')

<IPython.core.display.Math object>

Отсюда 
$$
\left[
\begin{aligned}
\dot{u} &= 0, \\
\dot{v} &= 0,
\end{aligned}
\right.
$$
то есть $u = const$ или $v = const$. Для прямого геликоида это либо спираль:

In [9]:
u0 = sp.Symbol('u_0', real=True)

fancy_display(r'\gamma_{u_0}(t) = ', r.subs({u: u0, v: t}), r',')

<IPython.core.display.Math object>

либо прямая:

In [10]:
v0 = sp.Symbol('v_0', real=True)

fancy_display(r'\ell_{v_0}(t) = ', r.subs({v: v0, u: t}))

<IPython.core.display.Math object>

## Задание 2

> Найти линии кривизны поверхности $$x = u^2 + v^2, \quad y = u^2 - v^2, \quad z = v.$$


In [11]:
u, v, t = sp.symbols('u,v,t', real=True)
a, b, c = sp.symbols('a:c', real=True, positive=True)

In [12]:
r = sp.Matrix([
    u**2 + v**2,
    u**2 - v**2,
    v
])

fancy_display(r'r = ', r, r'.')

<IPython.core.display.Math object>

In [13]:
def get_line_of_curvature(r):
    """
    Получает уравнение линии кривизны

    Возвращает левую часть ДУ 
    (FL - EM)du**2 + (GL - EN)dudv + (GM - FN)dv**2 = 0
    """

    du, dv = sp.symbols(r'\mathrm{d}u \mathrm{d}v', commutative=False)
    X = sp.Matrix([du, dv])
    
    (E, F, G), (L, M , N) = get_I_and_II(r)

    A = sp.Matrix([[E, F], [F, G]])
    B = sp.Matrix([[L, M], [M, N]])

    C = sp.Matrix.hstack(A * X, B * X)

    det = sp.expand(C.det())

    return det

In [14]:
det = get_line_of_curvature(r)

fancy_display(det, r' = 0.')

<IPython.core.display.Math object>

То есть линии кривизны — это координатные линии $u = const$ или $v = const$.

## Задание 3

> В области поверхности, не содержащей точек округления, доказать утверждение задачи № 828: координатные линии являются линиями кривизны тогда и только тогда, когда $$F = M = 0.$$

_Необходимость_

Для главных направлений $X_1, X_2$ должна быть верна выкладка:
$$
\begin{align*}
\mathrm{I}(S(X_1), X_2) = \mathbb{I}(X_1, X_2) &= \mathbb{I}(X_2, X_1) = \mathrm{I}(S(X_2), X_1)\\
\mathrm{I}(S(X_1), X_2) &= \mathrm{I}(S(X_2), X_1)\\
k_1 \mathrm{I}(X_1, X_2) &= k_2 \mathrm{I}(X_2, X_1)\\
\mathrm{I}(X_1, X_2) &= 0
\end{align*}
$$
Значит, $X_1 \perp X_2$ и 
$$
\mathbb{I}(X_1, X_2) = \mathrm{I}(S(X_1), X_2) = k_1 \mathrm{I}(X_1, X_2) = 0
$$

Выпишем первую и вторую квадратичные формы:

In [15]:
x1, x2 = sp.symbols('x^1, x^2', real=True)
y1, y2 = sp.symbols('y^1, y^2', real=True)
k = sp.Symbol('k', real=True)
E, F, G, L, M, N = sp.symbols('E, F, G, L, M, N', real=True)

A = sp.Matrix([[E, F], [F, G]])
B = sp.Matrix([[L, M], [M, N]])

X = sp.Matrix([x1, x2])
Y = sp.Matrix([y1, y2])

pairs_xy = [x1 * y1, x1 * y2, x2*y1, x2*y2]

II = sp.collect(sp.expand((X.T * B * Y)[0]), pairs_xy)
I  = sp.collect(sp.expand(k * (X.T * A * Y)[0]), pairs_xy)

fancy_display(r'\mathbb{I}(X, Y) = ', II)
fancy_display(r'\mathrm{I}(S(X), Y) = k \mathrm{I}(X, Y) =', I)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Раз $X_1$ и $X_2$ ортогональны, их можно сделать ортонормированным базисом в касательной плоскости.

In [16]:
X_1 = {x1: 1, x2: 0}
X_2 = {x1: 0, x2: 1}
Y_1 = {y1: 1, y2: 0}
Y_2 = {y1: 0, y2: 1}

I_at_X1_Y2  = I.subs(X_1).subs(Y_2)
II_at_X1_Y2 = II.subs(X_1).subs(Y_2)

I_at_X2_Y1  = I.subs(X_2).subs(Y_1)
II_at_X2_Y1 = II.subs(X_2).subs(Y_1)

fancy_display(r'X = ', X.subs(X_1), r', Y = ', Y.subs(Y_2), r'\implies', II_at_X1_Y2, r' = ', I_at_X1_Y2)
fancy_display(r'X = ', X.subs(X_2), r', Y = ', Y.subs(Y_1), r'\implies', II_at_X2_Y1, r' = ', I_at_X2_Y1)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Это равенство должно выполняться для всех главных кривизн. Значит, $M = F = 0$.

_Достаточность_

Если $F = M = 0$, то
$$
\begin{align*}
\mathrm{I} &= E \mathrm{d}u^2 + G \mathrm{d}v^2\\
\mathbb{I} &= L \mathrm{d}u^2 + N \mathrm{d}v^2
\end{align*}
$$
После нормировки получим
$$
\begin{align*}
\mathrm{I} &= \mathrm{d}u^2 + \mathrm{d}v^2\\
\mathbb{I} &= k_1 \mathrm{d}u^2 + k_2 \mathrm{d}v^2
\end{align*}
$$
Значит, координатные направления $X_1 = (1, 0)^\top, X_2 = (0, 1)^\top$ являются главными, то есть являются собственными векторами оператора Вейнгартена.

Что и требовалось.

## Задание 4

> Решить задачу № 832: доказать, что в области гиперболических точек линии кривизны в каждой точке делят пополам углы между асимптотическими линиями.

Точка гиперболическая $\implies$ есть два главных направления. Возьмём ортонормированный базис главных направлений. В нём вторая квадратичная форма будет иметь вид
$$
\mathbb{I} = \alpha^2 \mathrm{d}u^2 - \beta^2 \mathrm{d}v^2,
$$
где $\alpha^2, (-\beta^2)$ — главные кривизны.
Тогда 
$$
\mathbb{I} = 0 \iff (\alpha \mathrm{d}u - \beta \mathrm{d}v)(\alpha \mathrm{d}u + \beta \mathrm{d}v)= 0,
$$
откуда $u^\alpha \pm v^\beta = 0$ уравнения асимптотических линий. Относительно базисных векторов $e_1, e_2$ эти уравнения задают кривые, симметричные относительно $e_1, e_2$.

Что и требовалось.

## Задание 5

> На поверхности $z = 2x y$ найти асимптотические линии, проходящие через точку $P = (1, 1, 2)$.

In [17]:
u, v, t = sp.symbols('u,v,t', real=True)
a, b, c = sp.symbols('a:c', real=True, positive=True)

In [18]:
r = sp.Matrix([
    u,
    v,
    2 * u * v
])

u_P, v_P = 1, 1

fancy_display(r'r = ', r, r'.')

<IPython.core.display.Math object>

In [19]:
def get_asymptotic_directions(r):
    """
    Возвращает левую часть ДУ асимптотических линий
    """

    du, dv = sp.symbols(r'\mathrm{d}u \mathrm{d}v', commutative=False)
    X = sp.Matrix([du, dv])
    
    _, (L, M , N) = get_I_and_II(r)

    B = sp.Matrix([[L, M], [M, N]])
    C = sp.simplify(X.T * B * X)[0]
    
    return C

In [20]:
eq = get_asymptotic_directions(r)
fancy_display(eq, r' = 0.')

<IPython.core.display.Math object>

Значит, либо $u = 1, v = t$, либо $v = 1, u = t$.

In [21]:
fancy_display(r'\gamma_1(t) = ', r.subs({u: u_P, v: t}))

fancy_display(r'\gamma_2(t) = ', r.subs({u: t, v: v_P}))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Две прямые.